In [11]:
# =============================================================================
#  Fermi Velocity Prediction Pipeline  v2  — family + subfamily features
#  Input  : train_raw.csv, unique_m.csv, fermi_759.csv
#  Output : UCI_with_Fermi_Velocity.csv
#           (21,263 rows | 81 features + material + family + subfamily + Predicted_vF)
# =============================================================================

import pandas as pd
import numpy as np
import re
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# STEP 1 — Load data
# =============================================================================
df_train    = pd.read_csv("train_raw.csv")
df_unique_m = pd.read_csv("unique_m.csv")
df_fermi    = pd.read_csv("fermi_759.csv")   # 759 rows | 169 unique materials

# Keep ORIGINAL case for regex-based classification; lowercase only as merge key
df_unique_m['material_lower'] = df_unique_m['material'].str.strip().str.lower()
df_unique_m['material_orig']  = df_unique_m['material'].str.strip()
df_fermi['material']          = df_fermi['material'].str.strip().str.lower()

# lookup: lowercase → original case
orig_case_map = dict(zip(df_unique_m['material_lower'], df_unique_m['material_orig']))


In [12]:
# =============================================================================
# STEP 2 — Classification functions (operate on ORIGINAL case strings)
# =============================================================================
def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter → avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        # FIX-8: like has(), but requires the element's stoichiometric
        # coefficient to be >= threshold. No explicit number = coefficient 1.
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'   
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
       # FIX-1: MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'
 
    # FIX-2: heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'
 
    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        if has('Ce', mat) and has('In', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']):
            return 'cuprate_LSCO'
        if has('Bi', mat) and has('Sr', mat):
            return 'cuprate_BSCCO'
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        if has('Ba', mat) and (has('Y', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # FIX-4: Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
        # IRON-BASED
    if has_major('Fe', mat):
        if has('As', mat) and has('O', mat):
            return 'iron_1111'
        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # FIX-5: 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'
 
    # ── HEAVY FERMION ─────────────────────────────────────────────────────
        # HEAVY FERMION
    # FIX-3: Cu/Si added
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'
   
    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    # BOROCARBIDE
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'
    
    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
        # CONVENTIONAL A15
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga']) and not has_major('Fe', mat):
        return 'conventional_A15'  
    
    # ── CONVENTIONAL NbTi / transition metals ─────────────────────────────
    if has('Nb', mat):
        return 'conventional_NbTi'     
    
    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────    
    # FIX-6: CHALCOGENIDE before SIMPLE METALS
    # Sn0.97As0.022Te0.978, PbMo6S8 hit chalcogenide before Sn/Pb fires simple_metal
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    # SIMPLE METALS
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'
 
    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_NbTi':         'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# FIX-9: Flag low-confidence subfamily assignments
#
# conventional_NbTi in the 169-sample training set is really a catch-all for
# simple Nb-binary/ternary compounds (alloys, borides, carbides, simple oxides
# like LiNbO2, NbOS2). The 21,263-row UCI set also contains complex layered
# Ruddlesden-Popper niobates (e.g. KCa2Nb3O10, LiRbCa2Nb3O10 -- 4+ elements,
# Nb + O, no Ti). These have no analog in the training set, so predictions for
# them are extrapolation. Flag them rather than silently trusting the vF lookup.
# =============================================================================
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_NbTi' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False


In [13]:
# FIX-7: Single source of truth for vF — replaces raw CSV vF column
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          850000,
    'conventional_NbTi':         760000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}
 
def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)
 
 
# Sanity check — runs when cell executes
_tests = [
    ('Mg0.95Fe0.05B2',          'MgB2_type',              560000),
    ('MgB2',                    'MgB2_type',              560000),
    ('CeIn3',                   'heavy_fermion_115',        7595),
    ('CeCoIn5',                 'heavy_fermion_115',        7595),
    ('CeCu2Si2',                'heavy_fermion_Ce',         7595),
    ('Tl1.4Hg0.6Ba2Ca2Cu3O',   'cuprate_Tl_based',       320000),
    ('HgBa2CuO4',               'cuprate_Hg_based',       290000),
    ('Ca0.5Na0.5Fe2As2',        'iron_122',               121500),
    ('Sn0.97As0.022Te0.978',    'chalcogenide',            95000),
    ('PbMo6S8',                 'chalcogenide',            95000),
    ('Nd1.47La0.4Ce0.13Cu1O4',  'cuprate_electron_doped', 230000),
    ('Nd1.85Ce0.15CuO4',        'cuprate_electron_doped', 230000),
    ('YBa2Cu3O7',               'cuprate_YBCO',           273400),
    ('La1.85Sr0.15CuO4',        'cuprate_LSCO',           245000),
    ('Bi2Sr2CaCu2O8',           'cuprate_BSCCO',          304000),
    ('Nb3Sn',                   'conventional_A15',       850000),
    ('FeSe',                    'iron_11',                 95000),
    ('LiFeAs',                  'iron_111',               109000),
    ('LuNi2B2C',                'borocarbide',            165000),
]
 
print("=== STEP 2 SANITY CHECK ===\n")
_bugs = 0
for _mat, _exp_sf, _exp_vf in _tests:
    _got_sf = get_subfamily(_mat)
    _got_vf = get_vF(_mat)
    _sf_ok  = 'OK ' if _got_sf == _exp_sf else 'BUG'
    _vf_ok  = 'OK ' if _got_vf == _exp_vf else 'BUG'
    if 'BUG' in (_sf_ok, _vf_ok):
        _bugs += 1
    print(f"  {_sf_ok} {_vf_ok} | {_mat:40} | {_got_sf:30} | vF: {_got_vf:>8,}")
 
print(f"\n  >>> {'ALL CLEAR — 0 bugs' if _bugs == 0 else f'{_bugs} BUGS REMAINING'}\n")
 


=== STEP 2 SANITY CHECK ===

  OK  OK  | Mg0.95Fe0.05B2                           | MgB2_type                      | vF:  560,000
  OK  OK  | MgB2                                     | MgB2_type                      | vF:  560,000
  OK  OK  | CeIn3                                    | heavy_fermion_115              | vF:    7,595
  OK  OK  | CeCoIn5                                  | heavy_fermion_115              | vF:    7,595
  OK  OK  | CeCu2Si2                                 | heavy_fermion_Ce               | vF:    7,595
  OK  OK  | Tl1.4Hg0.6Ba2Ca2Cu3O                     | cuprate_Tl_based               | vF:  320,000
  OK  OK  | HgBa2CuO4                                | cuprate_Hg_based               | vF:  290,000
  OK  OK  | Ca0.5Na0.5Fe2As2                         | iron_122                       | vF:  121,500
  OK  OK  | Sn0.97As0.022Te0.978                     | chalcogenide                   | vF:   95,000
  OK  OK  | PbMo6S8                                  | chalcog

In [14]:

# =============================================================================
# STEP 3 — Build df_combine: all 21,263 superconductors
# =============================================================================
df_combine = df_train.merge(
    df_unique_m[['material_lower', 'material_orig']],
    left_index=True, right_index=True,
    how='inner'
)
df_combine['subfamily'] = df_combine['material_orig'].apply(get_subfamily)
df_combine['family']    = df_combine['material_orig'].apply(get_family)
df_combine['low_confidence'] = df_combine['material_orig'].apply(is_low_confidence_extrapolation)  # FIX-9
df_combine['material']  = df_combine['material_lower']
df_combine.drop(columns=['material_lower', 'material_orig'], inplace=True)

print(f"df_combine shape : {df_combine.shape}")
df_combine.to_csv("UCI_family_subFamily.csv", index=False)
print("Saved: UCI_family_subFamily.csv")

print("\nFamily distribution (df_combine):")
print(df_combine['family'].value_counts().to_string())

print("\nSubfamily distribution (df_combine):")
print(df_combine['subfamily'].value_counts().to_string())

print(f"\nLow-confidence (extrapolated) rows: {df_combine['low_confidence'].sum()} ({df_combine['low_confidence'].mean():.2%})")


df_combine shape : (21263, 86)
Saved: UCI_family_subFamily.csv

Family distribution (df_combine):
family
Cuprates                 10531
Conventional / Alloys     7118
Iron-based                1801
Chalcogenide              1164
Mg-B related               336
Heavy Fermion              313

Subfamily distribution (df_combine):
subfamily
cuprate_YBCO                 4367
other                        4245
cuprate_LSCO                 1702
cuprate_BSCCO                1691
chalcogenide                 1164
cuprate_other                1064
conventional_simple_metal     947
conventional_NbTi             923
iron_122                      702
conventional_A15              662
cuprate_Tl_based              644
cuprate_Hg_based              599
iron_1111                     581
cuprate_electron_doped        464
borocarbide                   341
MgB2_type                     336
iron_11                       315
heavy_fermion_Ce              166
heavy_fermion_115             143
iron_other     

In [ ]:
# =============================================================================
# STEP 4 — Build df_169: the 169-sample training set with family + subfamily features
# =============================================================================
df_759 = df_fermi.copy()

# Re-derive subfamily and family from original-case material names
# (overwrites the stored subfamily to ensure consistency with df_combine)
# df_759['material_orig'] = df_759['material'].map(orig_case_map)
# df_759['subfamily']     = df_759['material_orig'].apply(get_subfamily)
# df_759['family']        = df_759['material_orig'].apply(get_family)

# Add this at the end of STEP 4, after df_759 is built
df_759['material_orig'] = df_759['material'].map(orig_case_map)
df_759['vF (m/s)'] = df_759['material_orig'].apply(get_vF)   # ← reassign from VF_LOOKUP
df_759['subfamily'] = df_759['material_orig'].apply(get_subfamily)
df_759['family']    = df_759['material_orig'].apply(get_family)
df_759.drop(columns=['material_orig'], inplace=True)

print(f"\ndf_759 shape     : {df_759.shape}")
print(f"Unique materials : {df_759['material'].nunique()}")
print(f"Unique vF values : {df_759['vF (m/s)'].nunique()}")
print("\nSubfamily distribution (df_759):")
print(df_759['subfamily'].value_counts().to_string())
print(df_759.columns)
df_759.to_csv("Fermi_759_all_features.csv", index=False)


df_759 shape     : (759, 93)
Unique materials : 169
Unique vF values : 17

Subfamily distribution (df_759):
subfamily
cuprate_YBCO                 184
cuprate_BSCCO                 98
cuprate_Tl_based              78
cuprate_LSCO                  74
MgB2_type                     70
cuprate_electron_doped        38
borocarbide                   28
iron_111                      27
iron_122                      26
cuprate_Hg_based              24
iron_1111                     22
conventional_NbTi             21
iron_11                       14
iron_other                    12
chalcogenide                  10
conventional_A15               8
conventional_simple_metal      7
cuprate_other                  6
heavy_fermion_Yb               4
heavy_fermion_Ce               3
heavy_fermion_115              3
other                          2
Index(['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass',
       'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass',
      

In [16]:
import pandas as pd

df = pd.read_csv('Fermi_759_all_features.csv')

print(f"Before dedup: {df.shape}")
print(f"Unique materials: {df['material'].nunique()}")

# Keep one row per material — mean Tc, first for everything else
df_169 = df.groupby('material', as_index=False).agg(
    {
        'critical_temp' : 'mean',   # average across duplicate Tc measurements
        'vF (m/s)'      : 'first',  # vF is identical across duplicates
        **{col: 'first' for col in df.columns 
           if col not in ['material', 'critical_temp', 'vF (m/s)']}
    }
)

print(f"\nAfter dedup: {df_169.shape}")
print(f"Unique materials: {df_169['material'].nunique()}")

# Sanity check — vF should have zero variance per material
assert df_169['material'].nunique() == len(df_169), "Duplicates still exist!"
assert df_169['vF (m/s)'].nunique() > 1, "vF looks wrong"

print(f"\nvF range: {df_169['vF (m/s)'].min():,.0f} – {df_169['vF (m/s)'].max():,.0f} m/s")
print(f"Tc range: {df_169['critical_temp'].min():.1f} – {df_169['critical_temp'].max():.1f} K")

df_169.to_csv('Fermi_169_unique_v2.csv', index=False)
print("\nSaved: Fermi_169_unique_v2.csv ✓")

Before dedup: (759, 93)
Unique materials: 169

After dedup: (169, 93)
Unique materials: 169

vF range: 5,000 – 1,800,000 m/s
Tc range: 0.0 – 135.0 K

Saved: Fermi_169_unique_v2.csv ✓


In [17]:
# Check which materials had high Tc variance before averaging
tc_var = df.groupby('material')['critical_temp'].agg(['mean','std','count'])
high_var = tc_var[tc_var['std'] > 10].sort_values('std', ascending=False)
print("\nMaterials with high Tc variance (std > 10K):")
print(high_var)


Materials with high Tc variance (std > 10K):
                             mean        std  count
material                                           
tl0.8mo0.2sr2ca1cu2o7   50.000000  32.186954      3
tl2ba2cu1o6             58.888889  28.002258     45
mg1b1.75c0.25           16.250000  19.445436      2
bi2sr2ca0.6y0.4cu2o     72.700000  17.331186      3
pr0.6ca0.4ba2cu3o7      28.250000  15.202796      2
tl0.6sr2ca2cu3.4o       83.500000  14.849242      2
bi2sr2ca1cu2o8          77.628250  13.515140     80
y1ba2cu3o7              85.469791  13.194568    110
hg1ba2ca4cu5o           97.111111  13.119112      9
hg1ba2ca3cu4o          114.000000  12.000000      5
mg1b1.7c0.3             27.517500  11.933450      4


In [8]:

# =============================================================================
# STEP 5 — Define features and target
# =============================================================================
numeric_features     = [c for c in df_train.columns if c != 'critical_temp']  # all 81
categorical_features = ['family', 'subfamily']   # ← both family AND subfamily

X_train = df_169[numeric_features + categorical_features]
y_train = np.log10(df_169['vF (m/s)'].values)   # log10: 5k–1.8M → 3.7–6.3
groups  = df_169['material'].values                  # for GroupKFold

# print(f"\nX_train shape : {X_train.shape}")
print(f"y_train range : {y_train.min():.3f} – {y_train.max():.3f}  (log10 scale)\n")



y_train range : 3.699 – 6.255  (log10 scale)

